## Ejemplo de API Local en Colab (Servidor y Cliente en el mismo Notebook)

Aquí te muestro cómo puedes configurar una API local (usando Flask) y un cliente para interactuar con ella, todo dentro de este mismo notebook de Colab.

### 1. El Servicio (Servidor Flask)

Crearemos una pequeña aplicación Flask con un par de endpoints. La ejecutaremos en un hilo separado para que no bloquee la ejecución del notebook.

In [1]:
import threading
import time

import nest_asyncio
from flask import Flask, jsonify, request

# Aplicar nest_asyncio para permitir bucles de eventos anidados (necesario para Colab)
nest_asyncio.apply()

app = Flask(__name__)

# Un endpoint simple de bienvenida
@app.route('/')
def home():
    return '¡Hola desde tu API local!'

# Un endpoint que saluda a un nombre
@app.route('/greet/<name>')
def greet(name):
    return jsonify({
        'message': f'¡Hola, {name}!',
        'timestamp': time.time()
    })

# Un endpoint que suma dos números enviados por POST
@app.route('/sum', methods=['POST'])
def add_numbers():
    data = request.get_json()
    num1 = data.get('num1')
    num2 = data.get('num2')
    if isinstance(num1, (int, float)) and isinstance(num2, (int, float)):
        result = num1 + num2
        return jsonify({'sum': result})
    return jsonify({'error': 'Por favor, proporciona dos números válidos (num1, num2).'}), 400

def run_flask_app():
    # Ejecutar la aplicación Flask. Debug=True puede causar problemas con hilos.
    # Usamos threaded=True para que el servidor pueda manejar múltiples requests.
    app.run(port=5000, debug=False, use_reloader=False)

# Iniciar el servidor Flask en un hilo separado
flask_thread = threading.Thread(target=run_flask_app)
flask_thread.daemon = True # Esto asegura que el hilo se cerrará cuando el programa principal termine
flask_thread.start()

print('Servidor Flask iniciado en segundo plano en http://127.0.0.1:5000')
print('Espera unos segundos para que el servidor se inicialice completamente antes de hacer peticiones.')

Servidor Flask iniciado en segundo plano en http://127.0.0.1:5000
Espera unos segundos para que el servidor se inicialice completamente antes de hacer peticiones.


### 2. El Cliente (Peticiones a la API Local)

Esta celda de código funciona como el cliente que interactúa con la API Flask que hemos configurado y que se está ejecutando en segundo plano en este mismo entorno de Colab. Utiliza la librería `requests` de Python para enviar peticiones HTTP a la API y verificar su funcionamiento.

**Componentes clave y su función:**

*   **`import requests` y `import time`**: Importa las librerías necesarias. `requests` es fundamental para realizar peticiones web, y `time` se usa para introducir una pausa.
*   **`time.sleep(5)`**: Esta línea es crucial. Dado que el servidor Flask se inicia en un hilo separado, necesitamos darle un pequeño margen de tiempo (5 segundos en este caso) para que se inicialice completamente antes de que el cliente intente conectarse. Sin esta pausa, la primera petición podría fallar debido a que el servidor aún no está listo.
*   **`base_url = 'http://127.0.0.1:5000'`**: Define la URL base de nuestra API. `127.0.0.1` es la dirección IP de "localhost" (el propio equipo), y `5000` es el puerto en el que hemos configurado el servidor Flask para escuchar las peticiones.
*   **Pruebas de Endpoints (GET y POST)**:
    *   **`/` (GET)**: Envía una petición GET a la raíz de la API para probar el endpoint de bienvenida. Espera un mensaje simple en texto.
    *   **`/greet/<name>` (GET)**: Envía una petición GET al endpoint `/greet/Mundo`. Este endpoint espera un parámetro en la URL (el nombre) y devuelve una respuesta JSON con un saludo personalizado y una marca de tiempo.
    *   **`/sum` (POST) con datos válidos**: Realiza una petición POST al endpoint `/sum`. Para este tipo de petición, los datos se envían en el cuerpo de la petición en formato JSON (`'Content-Type': 'application/json'`). Se envían dos números (`num1`, `num2`) y se espera una respuesta JSON con la suma.
    *   **`/sum` (POST) con datos inválidos**: Demuestra cómo la API maneja errores. Aquí, se envía `num1` como una cadena de texto ("diez") en lugar de un número. El servidor Flask, según su lógica, debería detectar esto y devolver un error HTTP 400 (Bad Request) con un mensaje indicando el problema.
*   **Bloques `try-except requests.exceptions.ConnectionError`**: Cada petición está envuelta en un bloque `try-except` para capturar posibles errores de conexión. Esto es útil para diagnosticar si el servidor Flask no se inició correctamente o si hay algún problema de red.

In [2]:
import requests

# Dale un momento al servidor para que arranque
time.sleep(5)

base_url = 'http://127.0.0.1:5000'

print('--- Probando el endpoint / (GET) ---')
try:
    response = requests.get(base_url + '/')
    print(f'Estado: {response.status_code}')
    print(f'Respuesta: {response.text}')
except requests.exceptions.ConnectionError as e:
    print(f'Error de conexión: {e}. Asegúrate de que el servidor Flask se haya iniciado correctamente.')

print('\n--- Probando el endpoint /greet/<name> (GET) ---')
try:
    response = requests.get(base_url + '/greet/Mundo')
    print(f'Estado: {response.status_code}')
    print(f'Respuesta JSON: {response.json()}')
except requests.exceptions.ConnectionError as e:
    print(f'Error de conexión: {e}. Asegúrate de que el servidor Flask se haya iniciado correctamente.')

print('\n--- Probando el endpoint /sum (POST) ---')
try:
    headers = {'Content-Type': 'application/json'}
    data = {'num1': 10, 'num2': 25}
    response = requests.post(base_url + '/sum', json=data, headers=headers)
    print(f'Estado: {response.status_code}')
    print(f'Respuesta JSON: {response.json()}')
except requests.exceptions.ConnectionError as e:
    print(f'Error de conexión: {e}. Asegúrate de que el servidor Flask se haya iniciado correctamente.')

print('\n--- Probando el endpoint /sum (POST) con datos inválidos ---')
try:
    headers = {'Content-Type': 'application/json'}
    data = {'num1': '10', 'num2': 25}
    response = requests.post(base_url + '/sum', json=data, headers=headers)
    print(f'Estado: {response.status_code}')
    print(f'Respuesta JSON: {response.json()}')
except requests.exceptions.ConnectionError as e:
    print(f'Error de conexión: {e}. Asegúrate de que el servidor Flask se haya iniciado correctamente.')

127.0.0.1 - - [08/Aug/2026 19:48:41] "GET / HTTP/1.1" 200 -


127.0.0.1 - - [08/Aug/2026 19:48:41] "GET /greet/Mundo HTTP/1.1" 200 -


127.0.0.1 - - [08/Aug/2026 19:48:41] "POST /sum HTTP/1.1" 200 -


127.0.0.1 - - [08/Aug/2026 19:48:41] "POST /sum HTTP/1.1" 400 -


--- Probando el endpoint / (GET) ---
Estado: 200
Respuesta: ¡Hola desde tu API local!

--- Probando el endpoint /greet/<name> (GET) ---
Estado: 200
Respuesta JSON: {'message': '¡Hola, Mundo!', 'timestamp': 1786218521.642805}

--- Probando el endpoint /sum (POST) ---
Estado: 200
Respuesta JSON: {'sum': 35}

--- Probando el endpoint /sum (POST) con datos inválidos ---
Estado: 400
Respuesta JSON: {'error': 'Por favor, proporciona dos números válidos (num1, num2).'}


# JSONs

In [3]:
import json

# 1. json.loads(): Convierte una cadena JSON a un objeto Python
json_string = '{"nombre": "Alice", "edad": 30, "ciudades": ["Nueva York", "Londres"]}'
data_from_string = json.loads(json_string)
print("\n--- json.loads() ---")
print(f"Cadena JSON de entrada: {json_string}")
print(f"Objeto Python resultante: {data_from_string}")
print(f"Tipo del objeto: {type(data_from_string)}")
print(f"Accediendo a un elemento: {data_from_string['nombre']}")

# 2. json.dumps(): Convierte un objeto Python a una cadena JSON
python_dict = {
    'producto': 'Laptop',
    'precio': 1200.50,
    'disponible': True,
    'especificaciones': {
        'ram': '16GB',
        'almacenamiento': '512GB SSD'
    }
}
json_output_string = json.dumps(python_dict, indent=4) # indent para formato legible
print("\n--- json.dumps() ---")
print(f"Objeto Python de entrada: {python_dict}")
print(f"Cadena JSON resultante:\n{json_output_string}")
print(f"Tipo de la cadena: {type(json_output_string)}")

# 3. json.dump() y json.load(): Para lectura y escritura directa desde/hacia archivos

file_name = 'datos_ejemplo.json'

# Escribir en un archivo usando json.dump()
with open(file_name, 'w') as f:
    json.dump(python_dict, f, indent=4)
print("\n--- json.dump() (escritura en archivo) ---")
print(f"Objeto Python escrito en '{file_name}'. Puedes verificar el contenido del archivo.")

# Leer desde un archivo usando json.load()
with open(file_name) as f:
    data_from_file = json.load(f)
print("\n--- json.load() (lectura desde archivo) ---")
print(f"Contenido leído de '{file_name}': {data_from_file}")
print(f"Tipo del objeto leído: {type(data_from_file)}")

# Limpiar el archivo de ejemplo
#os.remove(file_name)
#print(f"\nArchivo '{file_name}' eliminado.")


--- json.loads() ---
Cadena JSON de entrada: {"nombre": "Alice", "edad": 30, "ciudades": ["Nueva York", "Londres"]}
Objeto Python resultante: {'nombre': 'Alice', 'edad': 30, 'ciudades': ['Nueva York', 'Londres']}
Tipo del objeto: <class 'dict'>
Accediendo a un elemento: Alice

--- json.dumps() ---
Objeto Python de entrada: {'producto': 'Laptop', 'precio': 1200.5, 'disponible': True, 'especificaciones': {'ram': '16GB', 'almacenamiento': '512GB SSD'}}
Cadena JSON resultante:
{
    "producto": "Laptop",
    "precio": 1200.5,
    "disponible": true,
    "especificaciones": {
        "ram": "16GB",
        "almacenamiento": "512GB SSD"
    }
}
Tipo de la cadena: <class 'str'>

--- json.dump() (escritura en archivo) ---
Objeto Python escrito en 'datos_ejemplo.json'. Puedes verificar el contenido del archivo.

--- json.load() (lectura desde archivo) ---
Contenido leído de 'datos_ejemplo.json': {'producto': 'Laptop', 'precio': 1200.5, 'disponible': True, 'especificaciones': {'ram': '16GB', 'a